In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import os, pathlib, stat, textwrap
import calendar
import datetime
from datetime import date
from tqdm import tqdm
from dask.distributed import Client, as_completed
from pathlib import Path

In [2]:
outpath = '/work/bk1450/b383184/Amazon/Mercator/data/variables_c/'

## URL
SSHfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-grid2D"

## environment
os.environ["NETRC"] = "/home/b/b383184/.netrc"

## mesh
ds_Zgr = xr.open_dataset('../data/Zgr_mesh.nc')
ds_Hgr = xr.open_dataset('../data/Hgr_mesh.nc')

## functions - Cut the data
lon0, lon1 = -95, 10
lat0, lat1 = -10, 30

# 1) Use T-point lon/lat
lonT = ds_Hgr.glamt.isel(t=0)
latT = ds_Hgr.gphit.isel(t=0)

# 2) Build boolean mask for your box
mask = (lonT >= lon0) & (lonT <= lon1) & (latT >= lat0) & (latT <= lat1)

# 3) Get index ranges
yy, xx = np.where(mask.values)

y0, y1 = int(yy.min()), int(yy.max())
x0, x1 = int(xx.min()), int(xx.max())

print(f"Spatial indices: x0={x0}, x1={x1}, y0={y0}, y1={y1}")

Spatial indices: x0=2305, x1=3565, y0=1374, y1=1873


In [3]:
##################################
## MERCATOR download function ####
##################################
def download_MERCATOR(url, varname, starts, ends, x0, x1, y0, y1, output_file,days):
    """
    Download MERCATOR data for given time slices.
    """
    parts = []
    for tt in tqdm(range(len(days)//2)): 
        da = (
            xr.open_dataset(url, engine="pydap", mask_and_scale=False, decode_cf=True)[varname]
            .sortby("time_counter")
            .isel(x=slice(x0, x1), y=slice(y0, y1))
            .sel(time_counter=slice(starts[tt], ends[tt]))
            .astype("float32")
            .load()
        )
        parts.append(da)
    
    da_all = xr.concat(parts, dim="time_counter")
    da_all = da_all.where(da_all != 9.96921e+36, np.nan)
    da_all.to_dataset(name=varname).to_netcdf(output_file, unlimited_dims=["time_counter"])
    print(f"Saved {output_file}")

In [4]:
##################################
## Helper function for dates #####
##################################
def glorys_days(start_date, end_date):
    """Generate GLORYS daily timestamps at 12:00."""
    return pd.date_range(start=start_date, end=end_date, freq="D") + pd.Timedelta(hours=12)

In [5]:
##################################
## Worker function for Dask ######
##################################
def run_one(year, month, SSHfiles, x0, x1, y0, y1, outpath):
    """
    Process one year-month: compute dates, download SSH data.
    """
    last_day = calendar.monthrange(year, month)[1]
    start_date = datetime.datetime(year, month, 1)
    end_date = datetime.datetime(year, month, last_day)
    
    # Compute days
    days = glorys_days(start_date.strftime("%Y-%m-%d"),
                       end_date.strftime("%Y-%m-%d"))
    starts = days[0::2]
    ends = days[1::2].tolist()
    ends[-1] = days[-1]
    
    SSH_out = f'SSH_{start_date.strftime("%Y-%m-%d")[:7]}c.nc'
    out_path = outpath + SSH_out
    
    # Check if file already exists (optional: skip if exists)
    if os.path.exists(out_path):
        print(f"Skipping {year}-{month:02d}, file exists: {out_path}")
        return f"Skipped {year}-{month:02d}"
    
    # Download
    download_MERCATOR(SSHfiles, "sossheig", starts, ends, x0, x1, y0, y1, out_path,days)
    return f"Done {year}-{month:02d} -> {out_path}"

In [6]:
##################################
## Main parallel execution ########
##################################
if __name__ == "__main__":
    # Start Dask client (adjust n_workers and threads_per_worker as needed)
    client = Client(processes=True, n_workers=8, threads_per_worker=1, memory_limit='16GB')
    print(client)
    print(client.dashboard_link)  # Open this in browser to monitor progress
    
    # Submit all year-month tasks
    futures = []
    for year in range(2007, 2014 + 1):
        for month in range(1, 12 + 1):
            fut = client.submit(run_one, year, month, SSHfiles, x0, x1, y0, y1, outpath)
            futures.append(fut)
    
    print(f"Submitted {len(futures)} tasks")
    
    # Collect results as they complete
    for fut in as_completed(futures):
        try:
            result = fut.result()
            print("[OK]", result)
        except Exception as e:
            print("[ERR]", e)
            import traceback
            traceback.print_exc()
    
    client.close()
    print("All tasks completed")

<Client: 'tcp://127.0.0.1:41705' processes=8 threads=8, memory=119.21 GiB>
http://127.0.0.1:8787/status
Submitted 96 tasks


  0%|                                                   | 0/15 [00:00<?, ?it/s]

Skipping 2007-04, file exists: /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2007-04c.nc
[OK] Skipped 2007-04
[OK] Skipped 2007-02
[OK] Skipped 2007-11
Skipping 2007-02, file exists: /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2007-02c.nc
Skipping 2007-11, file exists: /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2007-11c.nc
Skipping 2008-06, file exists: /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2008-06c.nc
[OK] Skipped 2008-06
[OK] Skipped 2007-06
[OK] Skipped 2008-04
[OK] Skipped 2008-09
Skipping 2007-06, file exists: /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2007-06c.nc
Skipping 2008-04, file exists: /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2008-04c.nc
Skipping 2008-09, file exists: /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2008-09c.nc


  0%|                                                   | 0/14 [00:00<?, ?it/s]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2007-08c.nc
[OK] Done 2007-08 -> /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2007-08c.nc
Saved /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2007-03c.nc
[OK] Done 2007-03 -> /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2007-03c.nc


...

[OK] Done 2014-09 -> /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2014-09c.nc


100%|██████████████████████████████████████████| 15/15 [01:17<00:00,  5.14s/it]


[OK] Done 2014-11 -> /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2014-11c.nc


100%|██████████████████████████████████████████| 15/15 [01:12<00:00,  4.83s/it]


[OK] Done 2014-12 -> /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2014-12c.nc
Saved /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2014-12c.nc
Saved /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2014-07c.nc
Saved /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2014-05c.nc
Saved /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2014-11c.nc
Saved /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2014-08c.nc
Saved /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2014-09c.nc
Saved /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2014-06c.nc
Saved /work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_2014-10c.nc
All tasks completed
